In [ ]:
import os
import sys
import glob
import pandas as pd

#avoid import errors
parent_dir=os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(parent_dir)

from scripts.CDS_machine import CDS_2_gffcomp
import scripts.get_busco_db as bob
import scripts.get_genetic_code as ggc
from scripts.counting_machine import write_counts, CATEGORIES


In [ ]:
raw_data="../../data/species"
query_email="wsxktjnfwcecxnotrf@gonrr.net"
busco_database="/no_backup/rg/references/busco_downloads"

!ls $raw_data

Babesia_duncani_323732
Chaetoceros_neogracilis_240364
Chlamydomonas_reinhardtii_3055
Cladocopium_goreaui_2562237
Conticribra_weissflogii_1577725
Cryptosporidium_parvum_Iowa_II_353152
Cyanidiococcus_yangmingshanensis_2690220
Cyanidioschyzon_merolae_strain_10d_280699
Cyanidioschyzon_merolae_strain_10D_280699
Cyanidium_caldarium_2771
Cylindrotheca_closterium_2856
Dunaliella_salina_3046
Durusdinium_trenchii_1381693
Eimeria_necatrix_51315
Emiliania_huxleyi_CCMP1516_280463
Entamoeba_histolytica_HM-1:IMSS_294381
Galdieria_yellowstonensis_3028027
Giardia_duodenalis_5741
Giardia_muris_5742
Haematococcus_lacustris_44745
Hamiltosporidium_tvaerminnensis_1176355
Leishmania_donovani_5661
Leishmania_infantum_JPCM5_435258
Micromonas_pusilla_CCMP1545_564608
Naegleria_gruberi_5762
Paramecium_bursaria_74790
Paramecium_tetraurelia_5888
Plasmodium_berghei_ANKA_5823
Plasmodium_falciparum_3D7_36329
Plasmodium_vivax_5855
Plasmodium_yoelii_5861
Symbiodinium_natans_878477
Symbiodinium_necroappetens_1628268
Tetr

# Merge reference and predicted annotations

Use agat, as there are no in-python tools that do all the conflict resolution

In [6]:
tr_sp=!ls $raw_data
pred_items = os.listdir("../results/pred") #only run on predicted items
pred_items=[tri.replace("_own.gff3", "") for tri in pred_items]
tr_sp = [species for species in tr_sp if species in pred_items]
print(len(tr_sp))


for src in ["own", "git"]:
    #checker(if things in file)
    fileWritten=False
    filepath=f"../job_commands/mergeAnn_{src}.txt"
    with open(filepath, "w") as f:
        print("mkdir -p ../results/merged", file=f)
        for sp in tr_sp:
            try:
                #if no  predictions exist, dont process this sspecies_source busco evaluation
                filename_git=f"../results/pred/{sp}_{src}*.gff3"
                found_files=glob.glob(filename_git) #supress error output

                if not found_files:
                    raise FileNotFoundError(f"There is no {src} predictions for this species")
                
                fileWritten=True #if any species has git parameters the job command files will be written
                
                #reference annotation
                RefAnn=!realpath ../training_data/species/$sp/CDS_*.gff
                RefAnn=RefAnn[0]

                #annotation prediction by source (own or git)
                pred=!realpath ../results/pred/$sp*_$src*.gff3
                pred=pred[0]

                #create folder to store outputs
                merged_res=f"../results/merged/{sp}_{src}.gff" #protein sequence output

                #write agat commands
                #per-task AGAT config so parallel array tasks don't collide on agat_config.yaml
                agat_cfg=f"agat_{sp}_{src}.yaml"
                expose_cmd=f"agat config --expose --output {agat_cfg} >/dev/null 2>&1"
                print(expose_cmd); print(expose_cmd, file=f)
                agat_merge=f"agat_sp_merge_annotations.pl --gff {RefAnn} --gff {pred} --config {agat_cfg} --out {merged_res}"

                print(agat_merge)
                print(agat_merge, file=f)
                clean_cmd=f"rm -f {agat_cfg}"
                print(clean_cmd); print(clean_cmd, file=f)

            except Exception as e:
                print(f"{sp} presents error: {e}")
                continue
            
    if not fileWritten: #if the file is not written, delete it
        os.remove(filepath)
            

4
agat_sp_merge_annotations.pl --gff /home/jj/Desktop/Data_science/CRG/TFM/projects/geneid-training/training_data/species/Cyanidiococcus_yangmingshanensis_2690220/CDS_Cyanidiococcus_yangmingshanensis_2690220_GenBank_GCA_013995675.1_11e137487fa95322bc32a4a414fe924a.gff --gff /home/jj/Desktop/Data_science/CRG/TFM/projects/geneid-training/results/pred/Cyanidiococcus_yangmingshanensis_2690220_own.gff3 --out ../results/merged/Cyanidiococcus_yangmingshanensis_2690220_own.gff
agat_sp_merge_annotations.pl --gff /home/jj/Desktop/Data_science/CRG/TFM/projects/geneid-training/training_data/species/Cyanidioschyzon_merolae_strain_10D_280699/CDS_Cyanidioschyzon_merolae_strain_10D_280699_GenBank_GCA_000091205.1_4d16b908b5f85da87b8c7b1d58441f2b.gff --gff /home/jj/Desktop/Data_science/CRG/TFM/projects/geneid-training/results/pred/Cyanidioschyzon_merolae_strain_10D_280699_own.gff3 --out ../results/merged/Cyanidioschyzon_merolae_strain_10D_280699_own.gff
agat_sp_merge_annotations.pl --gff /home/jj/Deskto

In [7]:
!ls ../results/merged

Cyanidiococcus_yangmingshanensis_2690220_own.gff
Cyanidioschyzon_merolae_strain_10D_280699_own.gff
Cyanidium_caldarium_2771_own.gff
Delisea_pulchra_40385_own.gff
Galdieria_partita_83374_own.gff
Galdieria_yellowstonensis_3028027_own.gff
Gracilaria_domingensis_172961_own.gff
Gracilariopsis_chorda_448386_own.gff
Porphyra_umbilicalis_2786_own.gff
Porphyridium_purpureum_35688_own.gff
Pyropia_vietnamensis_683352_own.gff
Rhodosorus_marinus_101924_own.gff


# Prediction analysis

## Reference annotation(CDS) and Prediction comparison

## BUSCO assessment

In [ ]:
#agat for gff>prot
tr_sp=!ls $raw_data
pred_items = os.listdir("../results/merged")
pred_items=[tri.replace("_own.gff", "") for tri in pred_items]
tr_sp = [species for species in tr_sp if species in pred_items]
print(len(tr_sp))


for src in ["own", "git"]:
    #checker(if things in file)
    fileWritten=False
    filepath=f"../job_commands/M_buscoScoring_{src}.txt"
    with open(filepath, "w") as f:
        print('cpus="${SLURM_CPUS_PER_TASK:-$(nproc)}"', file=f)
        print(f"mkdir -p ../results/summary/merged/busco_lineage ../results/summary/merged/busco_eukaryote", file=f)
        for sp in tr_sp:
            try:
                
                #if no prediction exists, dont process this sspecies_source busco evaluation
                filename_pred=f"../results/merged/{sp}_{src}.gff"
                found_files=glob.glob(filename_pred) #supress error output

                if not found_files:
                    raise FileNotFoundError(f"There is no {src} merged prediction for this species")
                
                fileWritten=True #if any species has git parameters the job command files will be written
                
                #reference sequence
                RefSeq=!realpath ../training_data/species/$sp/CLEAN_*.fna
                RefSeq=RefSeq[0]

                #merged annotation by source (own or git)
                merged=!realpath ../results/merged/$sp*_$src*.gff
                merged=merged[0]

                #create folder to store outputs
                prot_res=f"../results/specie_logs/{sp}/{sp}_merged_prot.fa" #merged protein sequence output
                busco_outPath=f"../results/merged_busco/" #busco comparisons output
                Lbusco_res=f"{sp}_{src}_Lbusco" #taxon-lineage busco out name
                Ebusco_res=f"{sp}_{src}_Ebusco" #eukaryota busco out name

                #get species taxon
                taxID=sp[sp.rfind("_")+1:]
                #find lineage
                busco_lineage=bob.taxon2lineage(taxID, query_email, f"{busco_database}/file_versions.tsv", "odb12")
                euk_lineage="eukaryota_odb12"
                #resolve the NCBI nuclear genetic code for this taxon (table 1 if unresolved)
                gcode=ggc.taxon2geneticcode(taxID, query_email) or 1

                #gff to protein command
                #per-task AGAT config so parallel array tasks don't collide on agat_config.yaml
                agat_cfg=f"agat_{sp}_{src}_merged.yaml"
                expose_cmd=f"agat config --expose --output {agat_cfg} >/dev/null 2>&1"
                print(expose_cmd); print(expose_cmd, file=f)
                TOprotein_cmd=f"agat_sp_extract_sequences.pl -g {merged} -f {RefSeq} -t cds -p --table {gcode} --config {agat_cfg} -o {prot_res}"

                print(TOprotein_cmd)
                print(TOprotein_cmd, file=f)
                clean_cmd=f"rm -f {agat_cfg}"
                print(clean_cmd); print(clean_cmd, file=f)
                #deduplicate protein IDs (AGAT can emit duplicate names that break BUSCO)
                ND_prot_res=prot_res.replace(".fa", "_ND.fa")
                rename_cmd=f"seqkit rename -n {prot_res} > {ND_prot_res}"
                print(rename_cmd)
                print(rename_cmd, file=f)
                
                Lbusco_cmd=f"busco -m protein \
                            -i {ND_prot_res} \
                            --download_path {busco_database} \
                            -l {busco_lineage} \
                            --cpu $cpus \
                            -f \
                            --opt-out-run-stats \
                            --out_path {busco_outPath} \
                            -o {Lbusco_res}" #--offline #autolineage creates errors
                Lbusco_cmd=Lbusco_cmd.replace("                             ", " ")

                Ebusco_cmd=f"busco -m protein \
                            -i {ND_prot_res} \
                            --download_path {busco_database} \
                            -l {euk_lineage} \
                            --cpu $cpus \
                            -f \
                            --opt-out-run-stats \
                            --out_path {busco_outPath} \
                            -o {Ebusco_res}" #--offline #autolineage creates errors
                Ebusco_cmd=Ebusco_cmd.replace("                             ", " ")

                print(Lbusco_cmd); print(Lbusco_cmd, file=f)
                print(Ebusco_cmd); print(Ebusco_cmd, file=f)

                #collect each run JSON into its own summary folder (isoquant naming)
                print(f"ln -vf {busco_outPath}{Lbusco_res}/short_summary.specific.*.json ../results/summary/merged/busco_lineage/{Lbusco_res}.json", file=f)
                print(f"ln -vf {busco_outPath}{Ebusco_res}/short_summary.specific.*.json ../results/summary/merged/busco_eukaryote/{Ebusco_res}.json", file=f)

            except Exception as e:
                print(f"{sp} presents error: {e}")
                continue

            
    if not fileWritten: #if the file is not written, delete it
        os.remove(filepath)
            

# Summary

The **regular** evaluation outputs are collected under `results/summary/regular/`: the busco cells above write the BUSCO summaries

In [ ]:
#metrics for the regular predictions -> results/summary/regular/counts/ per-species file + counts_summary.tsv.

results_dir = "../results"
summary_dir = f"{results_dir}/summary"
os.makedirs(summary_dir, exist_ok=True)

write_counts(results_dir, summary_dir, "merged", CATEGORIES["merged"])